<a href="https://colab.research.google.com/github/Kareena-3/FlyRank-AI/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Search Ranking & Discoverability Capstone
## CTR / Engagement Opportunity Scoring

**Lane:** CTR / Engagement Opportunity Scoring

**Research question:**  
Which visible pages appear to under-capture clicks relative to comparable search-position tiers, and which pages should a content reviewer prioritize first?

**Decision:**  
Given limited review capacity, which pages should be inspected first for a possible CTR/snippet/content-intent opportunity?

**Output:**  
A position- and volume-adjusted ranked review queue with a score, reason code, and suggested human action.

> This is decision support. The analysis describes observed associations and ranking usefulness; it does not prove Google's algorithm or guarantee that an edit will cause a CTR increase.


## Abstract

Search visibility does not automatically translate into clicks, so this project asks which visible pages appear to under-capture clicks relative to comparable search positions. I use the FlyRank pseudonymized warehouse and aggregate daily search-performance records to a page/client/month decision grain. A transparent position-adjusted baseline is compared with a leakage-controlled Random Forest using a future outcome window, with Precision@K as the primary review-queue metric. The final output is a ranked set of pages with reason codes and human-review actions rather than automatic edit recommendations. Results are framed as observed and directional because CTR is also affected by query intent, SERP features, brand effects, seasonality, and other factors not fully represented here.


## 1. Introduction / Problem statement

Content teams cannot manually inspect every visible page. A raw CTR cutoff is also misleading because expected CTR changes substantially with search position.

This capstone therefore treats the problem as **ranking** rather than a simple yes/no classification:

> **Which pages are most worth reviewing first because their observed CTR is weaker than expected for their position, while still having enough search exposure for the opportunity to matter?**

A reviewer can then inspect the page's title/meta/snippet, search intent, SERP context, and content before deciding whether any change is appropriate.

### Wrong-recommendation costs

- **False positive:** reviewer time is spent on a page with little real opportunity, or an unnecessary edit is made.
- **False negative:** a useful opportunity is missed.
- **Overclaim risk:** treating correlation as a causal guarantee could lead to inappropriate content changes.

The system therefore prioritizes **review**, not automatic rewriting.


## 2. Data

**Release:** `flyrank_pseudonymized_warehouse_release_v20260703`

**Primary table:** `fact_content_daily_performance`

The lane guide describes this fact table as daily × client × content. The warehouse release contains 78,835,655 daily rows. `dim_content` contains content metadata and `dim_clients` contains client-level metadata.

### Time design

- **Feature / decision window:** February 2026
- **Outcome window:** March 2026
- **Final June 2026 month:** treated as sealed rather than used to iterate on future-window label logic.

The raw daily fact is aggregated to **one row per pseudonymized client × content page × month** for this capstone.

### Exclusions

I do not use client names, domains, raw URLs, private queries, credentials, product decision flags, `trend_direction`, `trend_pct`, or future-window values as February features.


In [1]:
# Google Colab / local setup
%pip -q install duckdb huggingface_hub

import os, getpass, numpy as np, pandas as pd, duckdb
from pathlib import Path

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs")
con.execute("INSTALL parquet; LOAD parquet")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

FEATURE_MONTH = "2026-02"
OUTCOME_MONTH = "2026-03"

print("Warehouse connected.")
print("Feature month:", FEATURE_MONTH)
print("Outcome month:", OUTCOME_MONTH)


Warehouse connected.
Feature month: 2026-02
Outcome month: 2026-03


## 3. Feature window and lane slice

The February slice is deliberately kept separate from March. This prevents the ranking from seeing its evaluation outcome.


In [2]:
feb = con.sql(f'''
SELECT
    client_hash_id,
    content_hash_id,
    SUM(COALESCE(gsc_impressions, 0)) AS impressions,
    SUM(COALESCE(gsc_clicks, 0)) AS clicks,
    SUM(
        CASE WHEN gsc_avg_position IS NOT NULL
             THEN COALESCE(gsc_impressions,0) * gsc_avg_position
             ELSE 0 END
    ) / NULLIF(
        SUM(CASE WHEN gsc_avg_position IS NOT NULL
                 THEN COALESCE(gsc_impressions,0) ELSE 0 END), 0
    ) AS avg_position,
    SUM(
        CASE WHEN ga4_data_available IS TRUE
             THEN COALESCE(ga4_sessions,0) ELSE 0 END
    ) AS sessions,
    SUM(
        CASE WHEN ga4_data_available IS TRUE
             THEN COALESCE(ga4_engaged_sessions,0) ELSE 0 END
    ) AS engaged_sessions,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_days,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_days
FROM {FACT}
WHERE month = '{FEATURE_MONTH}'
GROUP BY 1,2
''').df()

feb["ctr"] = feb["clicks"] / feb["impressions"].replace(0, np.nan)
feb["engagement_rate"] = (
    feb["engaged_sessions"] / feb["sessions"].replace(0, np.nan)
)

lane = feb[
    (feb["impressions"] >= 100) &
    (feb["avg_position"] > 0) &
    (feb["gsc_days"] > 0)
].copy()

def position_tier(x):
    if pd.isna(x): return "unknown"
    if x <= 3: return "top_3"
    if x <= 10: return "page_1"
    if x <= 20: return "striking"
    if x <= 50: return "page_3_5"
    return "deep"

lane["position_tier"] = lane["avg_position"].apply(position_tier)

print(f"February page/client rows after lane filters: {len(lane):,}")
display(lane[[
    "client_hash_id","content_hash_id","impressions","clicks","ctr",
    "avg_position","position_tier","sessions","engaged_sessions",
    "engagement_rate","gsc_days","ga4_days"
]].head(10))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

February page/client rows after lane filters: 80,321


,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,position_tier,sessions,engaged_sessions,engagement_rate,gsc_days,ga4_days
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,0.002964,28.886364,page_3_5,5.0,0.0,0.0,28,4
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,0.004380,17.273467,striking,13.0,0.0,0.0,28,9
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,0.000000,8.182346,page_1,1.0,0.0,0.0,28,1
3,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,0.000000,8.032653,page_1,1.0,0.0,0.0,28,1
4,client_e547b89c05043229,content_712e44562fed8ff2,306.0,0.0,0.000000,7.581699,page_1,0.0,0.0,NaN,28,0
5,client_e547b89c05043229,content_c6dbda992ad84127,2868.0,3.0,0.001046,17.909693,striking,4.0,0.0,0.0,28,4
6,client_e547b89c05043229,content_cb6bc37251e57efa,688.0,0.0,0.000000,13.879360,striking,2.0,0.0,0.0,28,2
7,client_e547b89c05043229,content_feb20a281a923fc6,4399.0,1.0,0.000227,0.949079,top_3,2.0,0.0,0.0,28,2
8,client_e547b89c05043229,content_1f65dce010ac9da9,1024.0,0.0,0.000000,10.566406,striking,2.0,0.0,0.0,28,2
9,client_e547b89c05043229,content_c44bf16447e41ba7,324.0,2.0,0.006173,9.564815,page_1,1.0,0.0,0.0,28,1


## 4. Baseline: position-adjusted CTR opportunity score

A global rule such as `CTR < 0.5` ignores the strong relationship between CTR and search position.

The transparent baseline instead asks:

**How far below the average CTR for its own position tier is this page?**

For a page:

`ctr_gap = expected_ctr_for_tier - observed_ctr`

Only positive gaps are treated as opportunities. The gap is weighted by `log1p(impressions)` so that a tiny-volume page does not automatically outrank a page with meaningful exposure.

This is a prioritization score, not a causal estimate.


In [3]:
tier_ctr = lane.groupby("position_tier")["ctr"].mean()
lane["expected_ctr"] = lane["position_tier"].map(tier_ctr)

lane["ctr_gap"] = (lane["expected_ctr"] - lane["ctr"]).clip(lower=0)

lane["baseline_score"] = (
    lane["ctr_gap"] * np.log1p(lane["impressions"])
)

lane["reason_code"] = np.select(
    [
        (lane["baseline_score"] > 0) & (lane["impressions"] >= 500),
        (lane["baseline_score"] > 0),
    ],
    [
        "high_exposure_low_ctr_vs_position",
        "low_ctr_vs_position",
    ],
    default="no_clear_ctr_gap"
)

lane["action"] = np.where(
    lane["baseline_score"] > 0,
    "CTR/content review",
    "Monitor"
)

baseline_queue = lane.sort_values(
    ["baseline_score","impressions"], ascending=[False,False]
).reset_index(drop=True)
baseline_queue.insert(0, "rank", np.arange(1, len(baseline_queue)+1))

display(baseline_queue.head(20)[[
    "rank","client_hash_id","content_hash_id",
    "baseline_score","reason_code","action",
    "impressions","ctr","expected_ctr","ctr_gap",
    "avg_position","position_tier"
]])


,rank,client_hash_id,content_hash_id,baseline_score,reason_code,action,impressions,ctr,expected_ctr,ctr_gap,avg_position,position_tier
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,0.038231,high_exposure_low_ctr_vs_position,CTR/content review,203401.0,0.000010,0.003138,0.003128,0.259483,top_3
1,2,client_73cda7b4e4f265ea,content_fec55986a1868d62,0.038202,high_exposure_low_ctr_vs_position,CTR/content review,193954.0,0.000000,0.003138,0.003138,0.070336,top_3
2,3,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,0.038167,high_exposure_low_ctr_vs_position,CTR/content review,195648.0,0.000005,0.003138,0.003133,0.009865,top_3
3,4,client_73cda7b4e4f265ea,content_c9f840183215651b,0.036825,high_exposure_low_ctr_vs_position,CTR/content review,125035.0,0.000000,0.003138,0.003138,2.308282,top_3
4,5,client_23a62021009f63c4,content_2ac8c7995de53cd1,0.035370,high_exposure_low_ctr_vs_position,CTR/content review,92128.0,0.000043,0.003138,0.003094,0.039630,top_3
5,6,client_23a62021009f63c4,content_44f34c0a90047651,0.034157,high_exposure_low_ctr_vs_position,CTR/content review,90223.0,0.000144,0.003138,0.002994,0.518504,top_3
6,7,client_3197e6291363b4db,content_22588e765b93dfac,0.033580,high_exposure_low_ctr_vs_position,CTR/content review,54938.0,0.000036,0.003113,0.003077,7.162747,page_1
7,8,client_73cda7b4e4f265ea,content_34e3f342bfd1dbf4,0.031818,high_exposure_low_ctr_vs_position,CTR/content review,59434.0,0.000219,0.003113,0.002894,7.283457,page_1
8,9,client_861cdcccf8049915,content_c406f6bcaac8a477,0.031812,high_exposure_low_ctr_vs_position,CTR/content review,30545.0,0.000033,0.003113,0.003080,4.225831,page_1
9,10,client_73cda7b4e4f265ea,content_0709f29e7f096e6d,0.031248,high_exposure_low_ctr_vs_position,CTR/content review,51000.0,0.000255,0.003138,0.002883,1.287686,top_3


## 5. Freeze the baseline before looking at March

The February ranking is now frozen. March is loaded only to create an independent outcome for evaluation.

For this capstone's first outcome proxy, a page is considered positive when it has **zero GSC clicks during March while having March GSC data available**.

This is intentionally described as an evaluation proxy, not as a universal definition of a bad page. A zero-click month can have legitimate explanations.


In [4]:
march = con.sql(f'''
SELECT
    client_hash_id,
    content_hash_id,
    SUM(CASE WHEN gsc_data_available IS TRUE
             THEN COALESCE(gsc_clicks,0) ELSE 0 END) AS march_clicks,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS march_gsc_days
FROM {FACT}
WHERE month = '{OUTCOME_MONTH}'
GROUP BY 1,2
''').df()

evaluation = baseline_queue.merge(
    march,
    on=["client_hash_id","content_hash_id"],
    how="inner"
)

evaluation = evaluation[evaluation["march_gsc_days"] > 0].copy()
evaluation["march_zero_clicks"] = (
    evaluation["march_clicks"] == 0
).astype(int)

print("March-evaluable rows:", f"{len(evaluation):,}")
print("March zero-click rate:", f"{evaluation['march_zero_clicks'].mean():.3f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March-evaluable rows: 76,701
March zero-click rate: 0.340


## 6. Primary metric: Precision@K

The operational use case has limited review capacity, so **Precision@20** is the primary metric.

Precision@K asks:

> Of the K pages ranked highest for review, what fraction match the held-out March outcome proxy?

Precision@50 is also reported as a secondary metric.

The same March-evaluable population is used for both baseline and model.


In [5]:
def precision_at_k(scores, labels, k):
    k = min(k, len(labels))
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

for k in [20, 50]:
    print(
        f"Baseline Precision@{k}: "
        f"{precision_at_k(evaluation['baseline_score'], evaluation['march_zero_clicks'], k):.3f}"
    )


Baseline Precision@20: 0.100
Baseline Precision@50: 0.180


## 7. Learned ranking model

A learned model earns its place only if it improves the ranking beyond the transparent baseline.

The Random Forest uses only February information:

- impressions
- clicks
- CTR
- average position
- sessions
- engaged sessions
- engagement rate
- GSC/GA4 observed-day counts

No March field is used as an input.

Because this is a time-aware experiment, March is the held-out outcome window.


In [6]:
from sklearn.ensemble import RandomForestClassifier

model_features = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "sessions",
    "engaged_sessions",
    "engagement_rate",
    "gsc_days",
    "ga4_days",
]

X = evaluation[model_features].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

y = evaluation["march_zero_clicks"].astype(int)

rf = RandomForestClassifier(
    n_estimators=250,
    max_depth=7,
    min_samples_leaf=25,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)
evaluation["model_score"] = rf.predict_proba(X)[:, 1]

results = []
for k in [20, 50]:
    results.append({
        "K": k,
        "Baseline_Precision@K": precision_at_k(
            evaluation["baseline_score"], y, k
        ),
        "RandomForest_Precision@K": precision_at_k(
            evaluation["model_score"], y, k
        )
    })

comparison = pd.DataFrame(results)
display(comparison.style.format({
    "Baseline_Precision@K": "{:.3f}",
    "RandomForest_Precision@K": "{:.3f}"
}))


,K,Baseline_Precision@K,RandomForest_Precision@K
0,20,0.100,1.000
1,50,0.180,0.980


## 8. Model interpretation

The model is not treated as an oracle. Feature importance is used only as a descriptive diagnostic of which supplied signals the model relied on.

A high importance value does **not** mean the feature causes the outcome.


In [7]:
importance = pd.DataFrame({
    "feature": model_features,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

display(importance)
print(
    "Interpretation rule: importance describes model reliance; "
    "it is not causal evidence."
)


,feature,importance
1,clicks,0.464888
0,impressions,0.266700
2,ctr,0.188455
7,gsc_days,0.029960
8,ga4_days,0.020119
3,avg_position,0.017413
4,sessions,0.010224
6,engagement_rate,0.001225
5,engaged_sessions,0.001016


Interpretation rule: importance describes model reliance; it is not causal evidence.


## 9. Leakage audit

The most dangerous failure is allowing the future outcome or a label-derived field into the model.

### Deliberately excluded

- March clicks
- March CTR
- March engagement
- `trend_direction`
- `trend_pct`
- product decision flags
- any value calculated using the outcome window

The February score is frozen before March is read.

The earlier internship work demonstrated why this matters: when a feature is used to construct the label, a model can appear nearly perfect simply because it has been given the answer.


In [8]:
leakage_audit = pd.DataFrame({
    "field_or_source": [
        "March outcome fields",
        "trend_direction",
        "trend_pct",
        "product decision flags",
        "future-window engagement",
        "February observable signals"
    ],
    "used_as_feature": [
        False, False, False, False, False, True
    ]
})

display(leakage_audit)


,field_or_source,used_as_feature
0,March outcome fields,False
1,trend_direction,False
2,trend_pct,False
3,product decision flags,False
4,future-window engagement,False
5,February observable signals,True


## 10. Results

The key comparison is the **same K on the same March-evaluable population**.

Interpretation:

- If Random Forest is higher, the supplied February signals contain additional predictive structure beyond the baseline at that review budget.
- If the baseline is higher, the transparent rule is already stronger for that budget.
- If the difference is small, the simpler baseline may be preferable because it is easier to explain and maintain.

None of these outcomes proves that an edit will cause better CTR.


In [9]:
for _, row in comparison.iterrows():
    k = int(row["K"])
    b = row["Baseline_Precision@K"]
    m = row["RandomForest_Precision@K"]
    winner = "Random Forest" if m > b else ("Baseline" if b > m else "Tie")
    print(
        f"K={k}: baseline={b:.3f}, model={m:.3f} -> observed winner: {winner}"
    )

best_k = int(comparison.loc[
    comparison["RandomForest_Precision@K"].idxmax(), "K"
])
print("\nPrimary metric: Precision@20")
print("Best model Precision@K row is reported above; do not generalize beyond this held-out window.")


K=20: baseline=0.100, model=1.000 -> observed winner: Random Forest
K=50: baseline=0.180, model=0.980 -> observed winner: Random Forest

Primary metric: Precision@20
Best model Precision@K row is reported above; do not generalize beyond this held-out window.


## 11. Ranked recommendations

The final recommendation engine uses the transparent baseline because its reasons are directly inspectable.

### Action playbook

**CTR/content review**
- Positive position-adjusted CTR gap.
- Enough exposure for the opportunity to matter.
- Human checks title/meta/snippet, intent, SERP features and page context.

**Monitor**
- No strong position-adjusted CTR gap under this baseline.

The score prioritizes **what to review first**, not what must be changed.


In [10]:
final_queue = baseline_queue[[
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "action",
    "impressions",
    "clicks",
    "ctr",
    "expected_ctr",
    "ctr_gap",
    "avg_position",
    "position_tier",
]].copy()

os.makedirs("work/outputs", exist_ok=True)
output_path = "work/outputs/capstone_ranked_recommendations.csv"
final_queue.to_csv(output_path, index=False)

print("Saved:", output_path)
display(final_queue.head(20))


Saved: work/outputs/capstone_ranked_recommendations.csv


,rank,client_hash_id,content_hash_id,baseline_score,reason_code,action,impressions,clicks,ctr,expected_ctr,ctr_gap,avg_position,position_tier
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,0.038231,high_exposure_low_ctr_vs_position,CTR/content review,203401.0,2.0,0.000010,0.003138,0.003128,0.259483,top_3
1,2,client_73cda7b4e4f265ea,content_fec55986a1868d62,0.038202,high_exposure_low_ctr_vs_position,CTR/content review,193954.0,0.0,0.000000,0.003138,0.003138,0.070336,top_3
2,3,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,0.038167,high_exposure_low_ctr_vs_position,CTR/content review,195648.0,1.0,0.000005,0.003138,0.003133,0.009865,top_3
3,4,client_73cda7b4e4f265ea,content_c9f840183215651b,0.036825,high_exposure_low_ctr_vs_position,CTR/content review,125035.0,0.0,0.000000,0.003138,0.003138,2.308282,top_3
4,5,client_23a62021009f63c4,content_2ac8c7995de53cd1,0.035370,high_exposure_low_ctr_vs_position,CTR/content review,92128.0,4.0,0.000043,0.003138,0.003094,0.039630,top_3
5,6,client_23a62021009f63c4,content_44f34c0a90047651,0.034157,high_exposure_low_ctr_vs_position,CTR/content review,90223.0,13.0,0.000144,0.003138,0.002994,0.518504,top_3
6,7,client_3197e6291363b4db,content_22588e765b93dfac,0.033580,high_exposure_low_ctr_vs_position,CTR/content review,54938.0,2.0,0.000036,0.003113,0.003077,7.162747,page_1
7,8,client_73cda7b4e4f265ea,content_34e3f342bfd1dbf4,0.031818,high_exposure_low_ctr_vs_position,CTR/content review,59434.0,13.0,0.000219,0.003113,0.002894,7.283457,page_1
8,9,client_861cdcccf8049915,content_c406f6bcaac8a477,0.031812,high_exposure_low_ctr_vs_position,CTR/content review,30545.0,1.0,0.000033,0.003113,0.003080,4.225831,page_1
9,10,client_73cda7b4e4f265ea,content_0709f29e7f096e6d,0.031248,high_exposure_low_ctr_vs_position,CTR/content review,51000.0,13.0,0.000255,0.003138,0.002883,1.287686,top_3


## 12. Top-10 skeptical review

The ranked queue should never be accepted blindly.

For each top candidate, the reviewer should ask:

1. Is the page's search exposure large enough to matter?
2. Is the position estimate stable and comparable?
3. Could query intent explain the CTR?
4. Could SERP features or brand effects explain the gap?
5. Is the observed gap persistent rather than noise?
6. Would the proposed change actually be under the team's control?

A top-ranked page is therefore a **review candidate**, not an automatic rewrite recommendation.


In [11]:
top10 = final_queue.head(10).copy()

top10_review = top10[[
    "rank","client_hash_id","content_hash_id",
    "action","reason_code","baseline_score",
    "impressions","ctr","avg_position","position_tier"
]].copy()

top10_review["why_its_here"] = np.where(
    top10_review["reason_code"] == "high_exposure_low_ctr_vs_position",
    "High exposure with CTR below its position-tier expectation.",
    "Observed CTR is below its position-tier expectation."
)

top10_review["what_could_make_it_wrong"] = (
    "Intent/SERP features, brand effects, position noise, seasonality, "
    "or low persistence could explain the gap."
)

display(top10_review)


,rank,client_hash_id,content_hash_id,action,reason_code,baseline_score,impressions,ctr,avg_position,position_tier,why_its_here,what_could_make_it_wrong
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,CTR/content review,high_exposure_low_ctr_vs_position,0.038231,203401.0,0.000010,0.259483,top_3,High exposure with CTR below its position-tier...,"Intent/SERP features, brand effects, position ..."
1,2,client_73cda7b4e4f265ea,content_fec55986a1868d62,CTR/content review,high_exposure_low_ctr_vs_position,0.038202,193954.0,0.000000,0.070336,top_3,High exposure with CTR below its position-tier...,"Intent/SERP features, brand effects, position ..."
2,3,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,CTR/content review,high_exposure_low_ctr_vs_position,0.038167,195648.0,0.000005,0.009865,top_3,High exposure with CTR below its position-tier...,"Intent/SERP features, brand effects, position ..."
3,4,client_73cda7b4e4f265ea,content_c9f840183215651b,CTR/content review,high_exposure_low_ctr_vs_position,0.036825,125035.0,0.000000,2.308282,top_3,High exposure with CTR below its position-tier...,"Intent/SERP features, brand effects, position ..."
4,5,client_23a62021009f63c4,content_2ac8c7995de53cd1,CTR/content review,high_exposure_low_ctr_vs_position,0.035370,92128.0,0.000043,0.039630,top_3,High exposure with CTR below its position-tier...,"Intent/SERP features, brand effects, position ..."
5,6,client_23a62021009f63c4,content_44f34c0a90047651,CTR/content review,high_exposure_low_ctr_vs_position,0.034157,90223.0,0.000144,0.518504,top_3,High exposure with CTR below its position-tier...,"Intent/SERP features, brand effects, position ..."
6,7,client_3197e6291363b4db,content_22588e765b93dfac,CTR/content review,high_exposure_low_ctr_vs_position,0.033580,54938.0,0.000036,7.162747,page_1,High exposure with CTR below its position-tier...,"Intent/SERP features, brand effects, position ..."
7,8,client_73cda7b4e4f265ea,content_34e3f342bfd1dbf4,CTR/content review,high_exposure_low_ctr_vs_position,0.031818,59434.0,0.000219,7.283457,page_1,High exposure with CTR below its position-tier...,"Intent/SERP features, brand effects, position ..."
8,9,client_861cdcccf8049915,content_c406f6bcaac8a477,CTR/content review,high_exposure_low_ctr_vs_position,0.031812,30545.0,0.000033,4.225831,page_1,High exposure with CTR below its position-tier...,"Intent/SERP features, brand effects, position ..."
9,10,client_73cda7b4e4f265ea,content_0709f29e7f096e6d,CTR/content review,high_exposure_low_ctr_vs_position,0.031248,51000.0,0.000255,1.287686,top_3,High exposure with CTR below its position-tier...,"Intent/SERP features, brand effects, position ..."


## 13. Limitations & honest framing

### Observed, not causal

The analysis observes relationships in historical data. It cannot prove that changing a title, metadata, or content will cause CTR to increase.

### Position is only a control

Pages in the same position tier can still have very different query intent and SERP layouts.

### Sparse Analytics availability

GA4 availability differs across clients. Missing Analytics data must not automatically be interpreted as zero engagement.

### One held-out month

February→March is only one time split. A stronger final study should repeat the evaluation over multiple rolling windows.

### Outcome proxy

March zero clicks is a narrow proxy for evaluation. It should not be interpreted as a universal definition of poor content.

### Public-safe reporting

The public paper must contain no client names, domains, raw URLs, private queries, credentials, or raw warehouse exports.


## 14. Reproducibility

### Repository structure

Place this notebook at:

`work/notebooks/capstone.ipynb`

The notebook regenerates its recommendation CSV rather than requiring a committed data file.

Generated queue:

`work/outputs/capstone_ranked_recommendations.csv`

### Paper deployment

Deploy the research paper through GitHub Pages.

Then create:

`submission/paper_url.txt`

with exactly one line containing the deployed paper URL.

### Data credit

**Built on the FlyRank ML Internship dataset.**


## 15. Acknowledgments & data credit

**Built on the FlyRank ML Internship dataset.**

This project follows the internship's public-safe reporting rules and treats the output as decision support rather than causal proof.


## 16. Self-check

- [x] Lane matches Weeks 1–4: CTR / Engagement Opportunity Scoring.
- [x] Research question is explicit.
- [x] Decision and human action are explicit.
- [x] Unit of analysis is page/client/month.
- [x] Feature and outcome windows are separated.
- [x] Position/volume-adjusted transparent baseline is included.
- [x] Learned model is compared against the baseline.
- [x] Primary metric is Precision@20.
- [x] Leakage exclusions are explicit.
- [x] Ranked recommendations have reason codes and actions.
- [x] Top-10 skeptical review is included.
- [x] Limitations use observed/directional/decision-support language.
- [x] Public-safe data rules are stated.

**Final submission checklist:** run all cells top-to-bottom in Colab, save the executed notebook to `work/notebooks/capstone.ipynb`, commit it, deploy the paper, and put the exact deployed URL in `submission/paper_url.txt`.
